In [1]:
import re
import pandas as pd
import dotenv
import os
from neo4j import GraphDatabase

In [10]:
triplets_df_path = "../data/gfmag_sustainable_triplets.csv"
NEO4J_ENV = "../Neo4j-6b4d3211-Created-2026-01-22.txt"

In [3]:
triplets_df = pd.read_csv(triplets_df_path)

In [15]:
tester_df = triplets_df.head(10)

In [28]:
tester_df.rename(columns={"industry":"sector"}, inplace=True)

/tmp/ipykernel_2240480/976201657.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tester_df.rename(columns={"industry":"sector"}, inplace=True)


In [29]:
import os
import dotenv
from neo4j import GraphDatabase

def load_to_neo4j(df, env_file: str, batch_size: int = 1000):
    # load Aura credentials
    dotenv.load_dotenv(env_file)
    URI = os.getenv("NEO4J_URI")
    USER = os.getenv("NEO4J_USERNAME")
    PWD = os.getenv("NEO4J_PASSWORD")

    driver = GraphDatabase.driver(URI, auth=(USER, PWD))

    # batch insert function
    def batch_insert(tx, batch):
        for row in batch:
            rel_type = row['rel_type'].replace('`', '')  # sanitize
            query = f"""
            MERGE (e1:Entity {{name: $e1_name, type: $e1_type}})
            MERGE (e2:Entity {{name: $e2_name, type: $e2_type}})
            MERGE (e1)-[r:`{rel_type}`]->(e2)
            SET r.sector = $sector,
                r.url = $url,
                r.date = $date
            """
            tx.run(query,
                   e1_name=row['entity1'],
                   e1_type=row['entity1_type'],
                   e2_name=row['entity2'],
                   e2_type=row['entity2_type'],
                   sector=row['sector'],
                   url=row['url'],
                   date=str(row['date'])  # ensure string
                   )

    # main session loop
    with driver.session() as session:
        batch = []
        for _, row in df.iterrows():
            batch.append({
                'entity1': row['entity1'],
                'entity1_type': row['entity1_type'],
                'entity2': row['entity2'],
                'entity2_type': row['entity2_type'],
                'rel_type': row['rel_type'],
                'sector': row['sector'],
                'url': row['url'],
                'date': row['date'],
            })
            if len(batch) == batch_size:
                session.execute_write(batch_insert, batch)
                batch = []
        if batch:
            session.execute_write(batch_insert, batch)

    driver.close()
    print("Graph data loaded to Neo4j successfully.")


In [30]:
load_to_neo4j(df=tester_df, env_file=NEO4J_ENV)

Graph data loaded to Neo4j successfully.
